# 🚀 LLM Reasoning Benchmark Suite: Qwen2.5-7B-Instruct

Notebook đánh giá chuẩn hóa năng lực suy luận toán học trên **MATH-500** và **GSM8K** với 5 phương pháp:
1. **Direct**: Zero-shot Direct Answering.
2. **CoT**: Chain-of-Thought Natural Language Reasoning (Wei et al. 2022).
3. **PAL**: Program-Aided Language Models (Gao et al. ICML 2023) với Python tiêu chuẩn (`math`, `fractions`, `itertools`, `random`).
4. **SymCode**: Neurosymbolic Equation Solving với SymPy (ACL 2026).
5. **SymCode+**: SymCode tích hợp vòng lặp kiểm chứng độc lập & tự sửa lỗi (Traceback + Mathematical Verifier Feedback).

✨ **Tính năng nâng cao**:
- 📊 Phân rã Accuracy đa chiều: **Overall**, **by Subject**, **by Difficulty (Level)**, và **Subject × Difficulty**.
- 🛡️ **Auto-Checkpoint & Resume**: Tự động lưu kết quả liên tục và tiếp tục từ điểm dừng nếu bị ngắt kết nối.
- ⚡ **Fast Sandbox & Isolation**: Sandbox siêu tốc chạy trong RAM (<0.05s) với phân định rõ ràng quyền truy cập công cụ giữa PAL và SymCode.

In [ ]:
# 1. Smart Path Resolution & Environment Setup
import os
import sys
import glob

# Search recursively for the 'method' package in /kaggle/input and current workspace
def locate_directory(target_name):
    for search_root in ["/kaggle/input", ".", ".."]:
        if os.path.exists(search_root):
            for dirpath, dirnames, _ in os.walk(search_root):
                if target_name in dirnames:
                    return os.path.abspath(dirpath)
    return os.path.abspath(".")

ROOT_DIR = locate_directory("method")
if ROOT_DIR not in sys.path:
    sys.path.insert(0, ROOT_DIR)

print(f"[*] Detected Project Root: {ROOT_DIR}")

# Locate and install requirements
req_matches = glob.glob("/kaggle/input/**/requirements.txt", recursive=True) or ["requirements.txt"]
req_file = req_matches[0] if (req_matches and os.path.exists(req_matches[0])) else os.path.join(ROOT_DIR, "requirements.txt")

if os.path.exists(req_file):
    !pip install -q -r "{req_file}"
else:
    !pip install -q bitsandbytes accelerate transformers sympy datasets tqdm

In [ ]:
# 2. Import modules & Activate In-Memory Fast Sandbox + Verifier
import gc
import io
import re
import math
import time
import json
import queue
import random
import traceback
import contextlib
import concurrent.futures
import torch
import pandas as pd
import multiprocessing as mp

# Preload scientific modules in host process
import sympy
import sympy as sp
from sympy import (
    symbols, Eq, solve, simplify, Rational, Integer, Float,
    sympify, factorint, pi, sqrt, atan2, Matrix
)
import fractions
from fractions import Fraction
import itertools

from method import (
    LLMRunner,
    load_dataset_file,
    evaluate_direct_or_cot,
    evaluate_code_baseline,
    evaluate_symcode_plus,
    compute_metrics_table,
    save_benchmark_results,
    extract_boxed_content,
    extract_ground_truth,
    check_exact_match,
    verify_candidate_answer
)

# Fast Execution Sandbox with PAL / SymCode environment isolation
def _fast_run_code(code: str, mode: str = "symcode"):
    stdout_capture = io.StringIO()
    exec_globals = {
        "__builtins__": __builtins__,
        "math": math,
        "random": random,
        "fractions": fractions,
        "Fraction": Fraction,
        "itertools": itertools
    }
    if mode == "symcode":
        exec_globals.update({
            "sympy": sympy, "sp": sympy, "symbols": symbols, "Eq": Eq,
            "solve": solve, "simplify": simplify, "Rational": Rational,
            "Integer": Integer, "Float": Float, "sympify": sympify,
            "factorint": factorint, "pi": pi, "sqrt": sqrt, "atan2": atan2,
            "Matrix": Matrix
        })
    try:
        with contextlib.redirect_stdout(stdout_capture):
            exec(code, exec_globals)
        return {"status": "success", "stdout": stdout_capture.getvalue(), "traceback": None}
    except Exception:
        return {"status": "error", "stdout": stdout_capture.getvalue(), "traceback": traceback.format_exc()}

def fast_execute_code_safely(code: str, mode: str = "symcode", timeout: int = 15):
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as executor:
        future = executor.submit(_fast_run_code, code, mode)
        try:
            res = future.result(timeout=timeout)
        except concurrent.futures.TimeoutError:
            return {"status": "timeout", "stdout": "", "traceback": f"Execution timed out after {timeout}s.", "extracted_answer": None}
        except Exception as e:
            return {"status": "error", "stdout": "", "traceback": str(e), "extracted_answer": None}

    stdout = res.get("stdout", "")
    boxed_ans = extract_boxed_content(stdout)
    if boxed_ans is not None:
        res["extracted_answer"] = boxed_ans
    else:
        lines = [l.strip() for l in stdout.strip().split("\n") if l.strip()]
        res["extracted_answer"] = lines[-1] if lines else None
    return res

# Patch sandbox in method modules for guaranteed in-memory speed
import method.sandbox
import method.evaluator
method.sandbox.execute_code_safely = fast_execute_code_safely
method.evaluator.execute_code_safely = fast_execute_code_safely

print("[✓] Fast In-Memory Execution Sandbox successfully active!")
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device:", torch.cuda.get_device_name(0))

In [ ]:
# 3. 🔥 BẢNG ĐIỀU KHIỂN TÙY CHỈNH (CUSTOM BENCHMARK CONFIGURATION)

def find_data_file(subpath_pattern):
    matches = glob.glob(f"/kaggle/input/**/{subpath_pattern}", recursive=True)
    if matches:
        return os.path.abspath(matches[0])
    local_p = os.path.join(ROOT_DIR, subpath_pattern)
    if os.path.exists(local_p):
        return os.path.abspath(local_p)
    if os.path.exists(subpath_pattern):
        return os.path.abspath(subpath_pattern)
    return subpath_pattern

# ==========================================================================
# BƯỚC 1: CHỌN DATASET BẠN MUỐN CHẠY ("math500" hoặc "gsm8k")
# ==========================================================================
DATASET_CHOICE = "math500"  # Chọn "math500" hoặc "gsm8k"

# ==========================================================================
# BƯỚC 2: CHỌN CÁC PHƯƠNG PHÁP CẦN CHẠY (Thêm/bớt tùy thích)
# Các lựa chọn có sẵn: ["Direct", "CoT", "PAL", "SymCode", "SymCode+"]
# ==========================================================================
METHODS_TO_RUN = [
    "Direct",
    "CoT",
    "PAL",
    "SymCode",
    "SymCode+"
]

# ==========================================================================
# BƯỚC 3: CẤU HÌNH SỐ LƯỢNG MẪU & LỌC LEVEL
# - num_samples: 5 (test nhanh) | None (chạy full toàn bộ)
# - filter_levels: [1, 2, 3] (chỉ cho math500) | None (lấy tất cả level)
# ==========================================================================
NUM_SAMPLES = 5  # 🔥 Đổi thành None khi muốn chạy full toàn bộ
FILTER_LEVELS = [1, 2, 3] if DATASET_CHOICE == "math500" else None

# Auto-resolve dataset path
if DATASET_CHOICE == "math500":
    dataset_path = find_data_file("data/math500/test.jsonl")
    hf_fallback = "HuggingFaceH4/MATH-500"
    out_name = "math500_lvl1_3_results.json" if FILTER_LEVELS else "math500_results.json"
else:
    dataset_path = find_data_file("data/gsm8k/test.jsonl")
    hf_fallback = "openai/gsm8k"
    out_name = "gsm8k_results.json"

output_dir = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
output_file = os.path.join(output_dir, out_name)

# Nếu đang test <= 10 mẫu, xóa checkpoint cũ để chạy mới sạch sẽ
if NUM_SAMPLES is not None and NUM_SAMPLES <= 10 and os.path.exists(output_file):
    os.remove(output_file)
    print(f"[*] Reset fresh test output file: {output_file}")

CONFIG = {
    # Standardized model ID across the benchmark
    "model_id": "Qwen/Qwen2.5-7B-Instruct",
    "load_in_4bit": True,
    "max_new_tokens": 1024,
    "temperature": 0.0,
    "dataset_name": DATASET_CHOICE,
    "dataset_path": dataset_path,
    "hf_fallback": hf_fallback,
    "filter_levels": FILTER_LEVELS,
    "num_samples": NUM_SAMPLES,
    "methods_to_run": METHODS_TO_RUN,
    "code_exec_timeout": 15,
    "max_symcode_retries": 2, # Max retry budget for SymCode+ verifier loop
    "output_file": output_file,
    "save_every": 1 if NUM_SAMPLES and NUM_SAMPLES <= 10 else 5
}

print("=" * 60)
print(f"[*] Model: {CONFIG['model_id']}")
print(f"[*] Dataset: {CONFIG['dataset_name'].upper()} (Path: {CONFIG['dataset_path']})")
print(f"[*] Filter Levels: {CONFIG['filter_levels']}")
print(f"[*] Samples to evaluate: {CONFIG['num_samples'] if CONFIG['num_samples'] is not None else 'FULL'}")
print(f"[*] Methods to run: {CONFIG['methods_to_run']}")
print(f"[*] Results output: {CONFIG['output_file']}")
print("=" * 60)

In [ ]:
# 4. Dọn dẹp VRAM, Load Dataset & Khởi tạo Model 4-bit
if 'llm' in globals():
    del llm
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"[*] Cleared GPU memory. Current Allocated: {torch.cuda.memory_allocated() / (1024**2):.1f} MB")

dataset = load_dataset_file(
    CONFIG["dataset_path"],
    split="test",
    num_samples=CONFIG["num_samples"],
    filter_levels=CONFIG.get("filter_levels")
)

llm = LLMRunner(
    model_id=CONFIG["model_id"],
    load_in_4bit=CONFIG["load_in_4bit"],
    max_new_tokens=CONFIG["max_new_tokens"],
    temperature=CONFIG["temperature"]
)

In [ ]:
# 5. Execute Selected Methods with Fast Sandbox & Auto-Resume
out_f = CONFIG["output_file"]
save_n = CONFIG.get("save_every", 5)

if os.path.exists(out_f):
    try:
        with open(out_f, "r", encoding="utf-8") as f:
            benchmark_data = json.load(f)
    except Exception:
        benchmark_data = {"config": CONFIG, "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"), "results": {}, "summary": {}}
else:
    benchmark_data = {"config": CONFIG, "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"), "results": {}, "summary": {}}

for method in CONFIG["methods_to_run"]:
    if method in ["Direct", "CoT"]:
        benchmark_data["results"][method] = evaluate_direct_or_cot(
            method, dataset, llm, checkpoint_file=out_f, save_every=save_n
        )
    elif method in ["PAL", "SymCode"]:
        benchmark_data["results"][method] = evaluate_code_baseline(
            method, dataset, llm, timeout=CONFIG["code_exec_timeout"], checkpoint_file=out_f, save_every=save_n
        )
    elif method == "SymCode+":
        benchmark_data["results"]["SymCode+"] = evaluate_symcode_plus(
            dataset, llm, timeout=CONFIG["code_exec_timeout"], max_retries=CONFIG["max_symcode_retries"], checkpoint_file=out_f, save_every=save_n
        )
    else:
        print(f"[!] Unknown method: {method}. Skipping...")

In [ ]:
# 6. Multi-Dimensional Summary & Metrics Table (Overall + Subject x Difficulty)
summary = compute_metrics_table(benchmark_data["results"])
benchmark_data["summary"] = summary
save_benchmark_results(benchmark_data, CONFIG["output_file"])

# Display Overall DataFrame
df_overall = pd.DataFrame.from_dict({
    m: {
        "Accuracy (%)": v["accuracy_percent"],
        "Exact Match": f"{v['exact_match_count']}/{v['total_samples']}",
        "Avg Tokens": v["avg_generated_tokens"],
        "Avg Attempts": v["avg_attempts"],
        "Exec Success": v["execution_success_rate"],
        "Verif Pass": v["verification_success_rate"]
    }
    for m, v in summary.items()
}, orient="index")

print("\n--- [1] OVERALL METRICS TABLE ---")
display(df_overall)

# Display Subject x Difficulty Table for available methods
subj_diff_rows = []
for m, v in summary.items():
    for key, cell in v.get("by_subject_x_difficulty", {}).items():
        subj_diff_rows.append({
            "Method": m,
            "Subject": cell["subject"],
            "Difficulty": cell["difficulty"],
            "Correct": cell["correct"],
            "Total": cell["total"],
            "Accuracy (%)": cell["accuracy_percent"]
        })

if subj_diff_rows:
    print("\n--- [2] ACCURACY BY SUBJECT x DIFFICULTY TABLE ---")
    df_subj_diff = pd.DataFrame(subj_diff_rows)
    display(df_subj_diff)